# Try the personal-card-agent agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `middleware/` (customer-context seeding + tool-call logging). No `mda`
  CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.
- **Option C - test the data-visualization chart skill.** Builds the agent
  exactly as `server.py` does (same `backend`/`skills` wiring), needs a
  fuller set of env vars than A/B (see that section).
- **Option D - test the transfer-form skill.** Same wiring as Option C, a
  different skill/`ui["type"]` (`transfer_form` -- proposing a paydown for
  the customer's own credit card debt).
- **Option E - test the spending-jar skill.** Same wiring again, a third
  skill/`ui["type"]` pair (`create_jar`/`edit_jar` -- proposing a new or
  changed spending jar).

**Prerequisites**

```bash
uv sync --extra notebook
# fill AI_PLATFORM_API_KEY (and LANGSMITH_API_KEY for option B) in ../.env --
# every option below uses the same GreenNode VNG Cloud MaaS model as
# server.py, not OpenAI, so OPENAI_API_KEY is not needed here.
jupyter notebook notebooks/try_agent.ipynb
```

In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (middleware/, data/,
# instructions.md, .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("AI_PLATFORM_API_KEY"), (
    f"Set AI_PLATFORM_API_KEY in {REPO_ROOT / '.env'} first"
)

from langchain_openai import ChatOpenAI

from text_utils import extract_final_text

# Same GreenNode VNG Cloud MaaS endpoint/model server.py's _default_model()
# uses -- kept in sync manually since this notebook predates that switch and
# Option A/checklist below build their own agent without importing server.py
# (importing it would also require SERVER_API_KEY/MEMORY_ID/
# USER_PREFERENCE_STRATEGY_ID just for a quick local invoke -- see Option C).
LLM_BASE_URL = "https://maas-llm-aiplatform-hcm.api.vngcloud.vn/v1"
LLM_MODEL = os.environ.get("LLM_MODEL") or "z-ai/glm-5.2-hackathon"


def default_model() -> ChatOpenAI:
    return ChatOpenAI(base_url=LLM_BASE_URL, api_key=os.environ["AI_PLATFORM_API_KEY"], model=LLM_MODEL)


In [3]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

## Option A - quick local invoke (no `mda dev` needed)

In [ ]:
from deepagents import create_deep_agent

from middleware.audit import log_tool_calls
from middleware.seed_customer_context import SeedCustomerContext

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Thống kê số tiền tôi đã chi tiêu vào ăn uống"}
        ]
    }
)
print(extract_final_text(result["messages"][-1].content))

## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"personal-card-agent"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [ ]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "personal-card-agent"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(extract_final_text(chunk.data["messages"][-1]["content"]))

## Option C - test the data-visualization chart skill

Uses `server.build_agent()` directly (not a fresh `create_deep_agent(...)`
call like Option A) so the run actually exercises the same `backend=
CompositeBackend(...)`/`skills=[...]` wiring `server.py` uses in production --
a plain Option-A-style agent has no `skills=[...]`, so the skill would never
be loaded.

Importing `server` pulls in its full module-level env var checks (not just
`AI_PLATFORM_API_KEY`) -- make sure these are also set in `.env`:
`SERVER_API_KEY`, `MEMORY_ID`, `USER_PREFERENCE_STRATEGY_ID` (the last one
via `tools/memory_tools.py`, imported by `server.py`).

The question below explicitly asks for a `biểu đồ` (chart), echoing the
`data-visualization` skill's own description word-for-word -- the closest
thing to a guaranteed trigger this design allows (the model still decides,
there's no enforced schema). A missing/malformed `ui-json` block just
yields `ui is None`, it never breaks the call.

**Note (2026-09-03):** the response field was renamed from `chart` to `ui`
(a shared envelope for every widget type, discriminated by `ui["type"]`)
-- see `docs/api.md`'s breaking-change note and Option D below for the
second type, `transfer_form`.

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver, not the real GreenNode Memory checkpointer -- otherwise
# each re-run of this cell keeps appending to the SAME real, persistent
# thread ("test-chart") instead of starting a clean conversation, which
# silently changes the model's answer on later runs (it starts "remembering"
# earlier runs). Re-running this cell always starts a fresh, isolated agent.
chart_agent = server.build_agent(checkpointer=InMemorySaver())

result = chart_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Phân tích và vẽ biểu đồ so sánh chi tiêu các danh mục của tôi tháng này.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-chart", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

## Option D - test the transfer-form skill (trả nợ thẻ tín dụng)

Same wiring as Option C (`server.build_agent()`), just a different skill
(`skills/project/transfer-form/`) and a different `ui["type"]`
(`"transfer_form"` instead of `"chart"`).

This cell exercises **case (A)** only -- paying down the customer's own
credit card debt (`recipient`/`account_number` schema). See Option F below
for **case (B)**, transferring to a saved beneficiary via the real
`query_beneficiaries()` tool. The model verifies available credit itself
by reading `/customer_data.json` (`read_file`) -- there's no dedicated
tool for case (A), so expect only `read_file` calls in the output, no
tool call.

The question below asks about paying off a specific card's statement --
a direct match for the skill's description, similar to how Option C's
question echoes the chart skill's description.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason: avoids silently
# accumulating real history on GreenNode Memory across notebook re-runs.
transfer_agent = server.build_agent(checkpointer=InMemorySaver())

result = transfer_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tôi nên trả nợ thẻ Visa Signature bao nhiêu?",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-transfer-form", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

## Option E - test the spending-jar skill (tạo/chỉnh sửa hũ chi tiêu)

Same wiring as Option C/D (`server.build_agent()`), a third skill
(`skills/project/spending-jar/`) with two `ui["type"]` values:
`"create_jar"` (hũ chưa tồn tại) and `"edit_jar"` (hũ đã có, tính đến
đầu phiên -- xem `data/customers/CIF_0001.json`'s `jars` field, hoặc đã
được nhắc tới trong chính cuộc hội thoại này).

The model calls the `list_existing_jars()` tool (`tools/customer_data_tools.py`,
still mock data, keyed by customer_id) to check what already exists,
and reads `/customer_data.json` for `monthly_spending_summary` to ground
the proposed amount.

Two questions below, run in the same thread: first one should trigger
`create_jar` (CIF_0001 has no "Mua sắm" jar yet), the second should
trigger `edit_jar` for an already-existing jar (`jars` already has
"Giải trí" seeded in the mock data) even though it's never been
mentioned earlier in this conversation.

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason. This one matters
# even more here: the 2 questions below are DELIBERATELY run on the same
# thread (to test create_jar then edit_jar in one conversation) -- with the
# real checkpointer, re-running this cell keeps adding a 3rd, 5th, 7th...
# turn to the same real GreenNode thread each time, so "Tạo giúp tôi 1 hũ
# mua sắm" stops being a fresh request after the first run and starts
# correctly answering edit_jar instead (the model remembers offering it
# already) -- confusing if you don't expect it. InMemorySaver here means
# every notebook run is exactly the isolated 2-turn conversation it looks
# like in this cell.
jar_agent = server.build_agent(checkpointer=InMemorySaver())
JAR_THREAD = {"configurable": {"thread_id": "test-spending-jar", "actor_id": "CIF_0001"}}

for question in [
    "Tạo giúp tôi 1 hũ mua sắm hợp lý mỗi tháng.",  # expect create_jar
    "Tăng hũ giải trí của tôi lên 800 nghìn được không?",  # expect edit_jar (seeded in mock data, not mentioned above)
]:
    result = jar_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=JAR_THREAD,
    )
    answer = extract_final_text(result["messages"][-1].content)
    answer, ui = extract_ui_block(answer)
    print(f"Q: {question}")
    print("ANSWER:\n", answer)
    print("\nUI:\n", ui)
    print("-" * 60)

## Option F - test the transfer-form skill (chuyển tiền cho người nhận đã lưu)

Same wiring as Option D, testing **case (B)** this time: transferring to a
saved beneficiary. Unlike Option D, this calls the tool
`query_beneficiaries()` (`tools/customer_data_tools.py`), which hits the
**real** `pfm` backend over HTTP (`GET /api/beneficiaries?cif=`, see
`backend_docs/pfm-read-api.md`) -- this cell needs network access to the
deployed `pfm` endpoint, and the beneficiary names below are real data for
`CIF_0001` at the time this was written (re-run `query_beneficiaries()`
directly if the seed data has since changed).

Expect the model to call `query_beneficiaries()` first, find "Lan" in the
result, then propose a `transfer_form` using the new
`beneficiary_id`/`category` schema (NOT `recipient`/`account_number` --
see `docs/transfer-form.md`).


In [4]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason.
beneficiary_transfer_agent = server.build_agent(checkpointer=InMemorySaver())

result = beneficiary_transfer_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Chuyển tiền cho Lan 500 nghìn với nội dung Thanh toán tiền ăn.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-transfer-form-beneficiary", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)
assert ui is not None and ui["type"] == "transfer_form"
assert "beneficiary_id" in ui, "expected the pfm-lookup schema (beneficiary_id/category), not recipient/account_number"
assert ui["category"] in {
    "housing", "utilities", "subscriptions", "insurance", "dining",
    "transport", "shopping", "groceries", "entertainment", "health",
}


Model call: 5.76s
Calling tool: query_beneficiaries
Finished tool: query_beneficiaries (0.21s)
Model call: 3.48s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 6.77s
ANSWER:
 Mình đã điền sẵn form chuyển 500.000đ cho Nguyễn Thị Lan, bạn xem lại rồi bấm xác nhận nhé.

UI:
 {'type': 'transfer_form', 'beneficiary_id': 'b_stable_lan', 'amount': 500000, 'note': 'Thanh toan tien an', 'category': 'dining'}


## Option G - test the investment-guidance skill (short answer, decide-then-look-up)

Same wiring as Option D/F (`server.build_agent()`). Tests
`skills/project/investment-guidance/`: a goal-based investment question must get a
**4-5 sentence prose answer** (no tables/headings), grounded in the customer's real
income/spending/card debt, with any scenario folded into the conclusion ("Ngoài ra
bạn có thể cân nhắc ... nhưng cần khoảng ... vốn"). The flow is **decide first,
look up only when needed**: if the advice is "pay the card debt first" there is NO
MSB-product or news lookup; when a lookup is needed it is 1 search + 1 read, and
`middleware/tool_limits.py` hard-caps the calls per request. News is only used if
published within the last 4 days (older articles are refused by the tool); MSB pages
are fetched fresh (10-minute cache). Plain market-news questions are declined.

**Needs `TINYFISH_API_KEY` in `.env`** for the lookups.

Questions (fresh threads): plain news question -> declined; goal question with debt ->
no lookups; goal question as a customer with no jars; "which bank stock" -> at most
1 news search + 1 read.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
market_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "Hôm nay chứng khoán thế nào, vì sao VN-Index giảm?"),  # plain news question -> declined/redirected
        ("CIF_0001", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, tôi nên đầu tư gì?"),  # investment-guidance
        ("CIF_0001", "VN-Index giảm rồi, tôi nên mua cổ phiếu ngân hàng nào?"),  # no "buy X now"
        ("CIF_0002", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, tôi nên đầu tư gì?"),  # own numbers
    ]
):
    result = market_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-market-{i}", "actor_id": actor_id}},
    )
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", extract_final_text(result["messages"][-1].content))
    print("-" * 60)


## Option H - test the msb-products skill (sản phẩm MSB từ website chính thức)

Tools `search_msb_products` / `read_msb_product_page` (`tools/msb_product_tools.py`)
read MSB's own retail site (`msb.com.vn/khach-hang-ca-nhan/`), and
`list_msb_documents` / `read_msb_document` read MSB's official PDFs (fee
schedules, loan rates, promo terms), all through TinyFish Search + Fetch -- same `TINYFISH_API_KEY` as Option G. Only that URL prefix is
readable (host/path allowlist enforced in code). Known limitation: the deposit
rate TABLE on the MSB site is rendered client-side, so the agent can only quote
rates that appear as text on a page (e.g. "sinh lời đến 6.9%/năm") and otherwise
points the customer to MSB's rate page / app.

Expect: answers cite the MSB page (name + URL) and remind that terms are
confirmed in the app / with an advisor; a goal-based investment question now
also names suitable MSB products it found; a "which bank pays most" question is
still refused (no rival-bank rates, no advice to bank elsewhere).

PDF note: the 4 deposit-rate PDFs are scanned images (no text layer) so they
cannot be read -- the agent must say so and point to the rate page / app.
Loan-rate and fee PDFs are readable.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
msb_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "MSB có chứng chỉ tiền gửi không? Lãi bao nhiêu, điều kiện thế nào?"),
        ("CIF_0002", "MSB có thẻ tín dụng nào hoàn tiền không?"),
        ("CIF_0002", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, MSB có sản phẩm nào phù hợp?"),
        ("CIF_0001", "Phí thường niên thẻ ghi nợ MSB là bao nhiêu?"),  # PDF: bieu_phi
        ("CIF_0001", "Lãi suất vay tại MSB hiện nay thế nào?"),  # PDF: lai_suat_vay
        ("CIF_0002", "Lãi suất tiết kiệm 12 tháng của MSB là bao nhiêu?"),  # scanned PDF -> point to rate page
        ("CIF_0001", "Lãi suất tiết kiệm 12 tháng các ngân hàng là bao nhiêu? Tôi nên gửi ngân hàng nào?"),  # refused
    ]
):
    result = msb_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-msb-{i}", "actor_id": actor_id}},
    )
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", extract_final_text(result["messages"][-1].content))
    print("-" * 60)


## Option I - test the investment scenario projection (+ chart)

Tool `project_investment_growth` (`tools/investment_tools.py`, pure arithmetic)
computes, under a HYPOTHETICAL monthly return (default +2%/month -- a
product-owner assumption, not a measured market average): the capital needed to
earn a target gain by 31/12, and the value of a chosen capital at each
month-end for three scenarios (+r, flat 0%, -r). It also returns a ready-made
`ui-json` line chart. The agent must call it a "kịch bản giả định", always show
the flat and losing scenarios, and NOT assume the customer has a jar named
"tiết kiệm" -- capital comes from an amount the customer states, or is
estimated from balances (account + `list_existing_jars` total - emergency fund
- card debt), with 50% of the idle money proposed by default.

Answers are 4-5 sentences (charts only if the customer asks). Four questions, fresh threads: (1) target only, no stated amount; (2) a stated
amount + target; (3) another customer with no jars; (4) a large stated amount
(target reachable only in the +r scenario). Chart blocks are split out by
`extract_ui_block`, so `ui` prints the validated chart.


In [6]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
scenario_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "MSB Có sản phẩm gì để đầu tư không"),
        # ("CIF_0002", "Tôi có 60 triệu nhàn rỗi, muốn có thêm 10 triệu đến cuối năm, tôi nên đầu tư gì?"),
        # ("CIF_0003", "Tôi muốn có thêm 10 triệu đến cuối năm, tôi nên đầu tư gì?"),
        # ("CIF_0003", "Tôi có 400 triệu nhàn rỗi, muốn có thêm 10 triệu đến cuối năm, nên đầu tư thế nào?"),
    ]
):
    result = scenario_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-scenario-{i}", "actor_id": actor_id}},
    )
    answer, ui = extract_ui_block(extract_final_text(result["messages"][-1].content))
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", answer)
    print("UI (chart):\n", ui)
    print("-" * 60)


Model call: 6.63s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 5.26s
Calling tool: search_msb_products
Calling tool: search_msb_products
Calling tool: search_msb_products
Finished tool: search_msb_products (1.23s)
Finished tool: search_msb_products (1.57s)
Finished tool: search_msb_products (1.87s)
Model call: 6.48s
Calling tool: search_msb_products
Calling tool: read_msb_product_page
Calling tool: read_msb_product_page
Finished tool: search_msb_products (1.25s)
Finished tool: read_msb_product_page (3.96s)
Finished tool: read_msb_product_page (4.82s)
Model call: 5.11s
Calling tool: search_msb_products
Calling tool: search_msb_products
Finished tool: search_msb_products (1.26s)
Finished tool: search_msb_products (1.26s)
Model call: 5.27s
Calling tool: read_msb_product_page
Finished tool: read_msb_product_page (2.82s)
Model call: 20.76s
[CIF_0001] Q: MSB Có sản phẩm gì để đầu tư không
ANSWER:
 MSB có 3 nhóm sản phẩm để bạn đầu tư/sinh lời:

**1. Tiết kiệm & tiền g

## Verification checklist (local, rows 1-9 and 11 only — row 10 needs Studio, see docs/superpowers/plans/2026-08-30-personal-card-agent-verification.md)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

verify_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
    checkpointer=MemorySaver(),
)

CHECKLIST_QUESTIONS = [
    "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?",
    "Giao dịch lớn nhất tuần vừa rồi là gì?",
    "Liệt kê các giao dịch trên 2 triệu trong tháng 7",
    'Danh mục nào đang "ngốn" tiền của tôi nhất?',
    "Tôi có đang trả lãi thẻ tín dụng không, và vì sao?",
    "Tôi nên dùng thẻ tín dụng hay ghi nợ cho khoản chi 5 triệu này?",
    "Ngày sao kê và ngày đến hạn thanh toán của thẻ tôi là khi nào?",
    "Tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Ngân sách của tôi là 15 triệu/tháng.",
    "Vậy tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Khoá thẻ của tôi giúp tôi.",
]
THREAD_CONFIG = {"configurable": {"thread_id": "notebook-verify-1"}}

for question in CHECKLIST_QUESTIONS:
    result = verify_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=THREAD_CONFIG,
    )
    print(f"Q: {question}\nA: {extract_final_text(result['messages'][-1].content)}\n{'-' * 60}")

print(
    "Row 10 (cross-thread durable budget memory) cannot be tested locally — "
    "run mda dev . and re-ask the budget question in a NEW Studio thread instead."
)
